# PyTorch Refresher for Building an Inference Engine from Scratch

This notebook is a working refresher, not a general PyTorch tutorial. Every
concept here was picked because you will use it directly in `model.py`,
`kv_cache.py`, and `sampling.py` over the next few weeks. Each section ends
with a **"Where this shows up"** note pointing at the exact place in the
project.

Run every cell top to bottom — later cells depend on earlier ones (imports,
helper tensors).

Sections:
1. Tensors: creation, dtype, device
2. Shapes: view / reshape / permute / transpose / contiguous
3. Broadcasting
4. Indexing and slicing
5. Matrix multiplication: `@`, `matmul`, `bmm`, `einsum`
6. Autograd: `requires_grad`, `backward`, `no_grad`, `inference_mode`
7. `nn.Module`, `nn.Parameter`, buffers (ties directly to the `RMSNorm` you already wrote)
8. Numerical precision: fp32 vs bf16/fp16, and why reductions get upcast
9. Softmax and masking (the core of attention)
10. The multi-head reshape pattern: splitting and merging heads
11. `repeat_interleave` / `expand` for GQA's shared KV heads
12. Complex numbers for RoPE: `polar`, `view_as_complex`, `view_as_real`
13. `register_buffer` for precomputed, non-trainable state
14. Preallocated tensors + in-place writes (the KV-cache pattern)
15. Sampling primitives: `topk`, `sort`, `cumsum`, `multinomial`
16. Inspecting and loading real pretrained weights (`state_dict`)
17. Apple Silicon / MPS: what works, what doesn't, and the fallback pattern
18. Testing numerics: `torch.testing.assert_close` and why `atol`/`rtol` matter
19. A preview of `torch.compile` and CUDA graphs (Week 2 territory)

## 0. Setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

print("torch version:", torch.__version__)
print("MPS built:", torch.backends.mps.is_built())
print("MPS available:", torch.backends.mps.is_available())

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

torch version: 2.14.0
MPS built: True
MPS available: True
Using device: mps


## 1. Tensors: creation, dtype, device

A `torch.Tensor` is a multi-dimensional array like a NumPy array, but with
two extra superpowers you'll use constantly: it can live on an accelerator
(`device`), and it can track gradients (`requires_grad`, Section 6).

Three things define a tensor's identity: its **shape**, its **dtype**
(the numeric type each element is stored as), and its **device** (which
physical memory it lives in). An operation between two tensors generally
requires matching dtype and device — this is one of the most common sources
of runtime errors when wiring up a model.

In [2]:
# Creation
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.zeros(2, 3)
c = torch.ones(2, 3)
d = torch.randn(2, 3)          # standard normal
e = torch.arange(6).reshape(2, 3)

print("a:", a, a.dtype, a.shape)
print("d:\n", d)
print("e:\n", e, e.dtype)

a: tensor([1., 2., 3.]) torch.float32 torch.Size([3])
d:
 tensor([[ 1.5410, -0.2934, -2.1788],
        [ 0.5684, -1.0845, -1.3986]])
e:
 tensor([[0, 1, 2],
        [3, 4, 5]]) torch.int64


In [3]:
# dtype matters a lot for inference: weights are usually bf16/fp16 on GPU,
# but some reductions (like RMSNorm's mean-of-squares) are done in fp32 for
# numerical stability -- you already applied this in your RMSNorm.
x_fp32 = torch.randn(3, dtype=torch.float32)
x_bf16 = x_fp32.to(torch.bfloat16)
x_fp16 = x_fp32.to(torch.float16)

print("fp32:", x_fp32)
print("bf16:", x_bf16)   # bf16 has the same exponent range as fp32 but fewer mantissa bits -> coarser values
print("fp16:", x_fp16)

# Casting back and forth loses precision -- this round-trip is NOT a no-op:
print("fp32 -> bf16 -> fp32, same as original?", torch.equal(x_fp32, x_bf16.to(torch.float32)))

fp32: tensor([ 0.4033,  0.8380, -0.7193])
bf16: tensor([ 0.4043,  0.8398, -0.7188], dtype=torch.bfloat16)
fp16: tensor([ 0.4033,  0.8379, -0.7192], dtype=torch.float16)
fp32 -> bf16 -> fp32, same as original? False


In [4]:
# Device placement. `.to(device)` moves (copies) a tensor; ops require both
# operands on the same device.
x_cpu = torch.randn(3)
x_dev = x_cpu.to(device)
print(x_cpu.device, "->", x_dev.device)

try:
    x_cpu + x_dev  # will error if device != "cpu"
except RuntimeError as err:
    print("Expected error mixing devices:", err)

cpu -> mps:0
Expected error mixing devices: Expected all tensors to be on the same device, but found at least two devices, mps:0 and cpu!


**Where this shows up:** every weight matrix and activation in your model
has a dtype (usually bf16 for real checkpoints, fp32 while debugging) and a
device. `perf_model.py`'s byte-counting math (Week 1, Day 6) is directly
"how many elements times how many bytes per element for this dtype."

## 2. Shapes: `view`, `reshape`, `permute`, `transpose`, `contiguous`

A tensor's data sits in memory as one flat buffer. Its `shape` and `stride`
(how many elements to skip to move one step along each dimension) describe
how to *interpret* that flat buffer as a multi-dimensional array.

- **`view`**: reinterprets the same memory with a new shape. Requires the
  data to already be laid out compatibly (contiguous, in the relevant
  sense) — fails otherwise.
- **`reshape`**: like `view`, but will silently copy if the data isn't
  laid out compatibly. Safer, marginally more expensive when a copy is
  needed.
- **`transpose(dim0, dim1)`** / **`permute(*dims)`**: swap axes *without*
  moving data — they only change the strides. This makes the tensor
  non-contiguous.
- **`.contiguous()`**: forces a real memory copy into standard row-major
  layout. You need this before some ops (and before `.view()`) if you've
  just transposed/permuted.

This exact sequence — reshape to split heads, transpose so the head
dimension is second, run attention, transpose back, reshape to merge heads
— is precisely what you'll write for GQA attention.

In [5]:
x = torch.arange(24).reshape(2, 3, 4)   # (batch=2, seq=3, hidden=4)
print("x.shape:", x.shape, "x.stride():", x.stride(), "contiguous?", x.is_contiguous())

# transpose seq and hidden -- no data movement, just relabels strides
xt = x.transpose(1, 2)
print("xt.shape:", xt.shape, "xt.stride():", xt.stride(), "contiguous?", xt.is_contiguous())

# view on the transposed tensor fails: the memory isn't laid out for it
try:
    xt.view(2, 4 * 3)
except RuntimeError as err:
    print("view() fails on a non-contiguous tensor:", err)

# reshape() works anyway (it copies under the hood)
print("reshape works:", xt.reshape(2, 4 * 3).shape)

# .contiguous() then view() also works, and is the idiomatic pattern in
# attention code: transpose -> contiguous -> view
print("contiguous().view() works:", xt.contiguous().view(2, 4 * 3).shape)

x.shape: torch.Size([2, 3, 4]) x.stride(): (12, 4, 1) contiguous? True
xt.shape: torch.Size([2, 4, 3]) xt.stride(): (12, 1, 4) contiguous? False
view() fails on a non-contiguous tensor: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.
reshape works: torch.Size([2, 12])
contiguous().view() works: torch.Size([2, 12])


In [6]:
# The exact split-heads pattern you'll use for multi-head attention:
batch, seq_len, hidden_size = 2, 5, 16
n_heads, head_dim = 4, 4   # hidden_size == n_heads * head_dim here

x = torch.randn(batch, seq_len, hidden_size)

# (B, T, H*D) -> (B, T, H, D) -> (B, H, T, D)
# We split hidden into (heads, head_dim), THEN move heads before seq so that
# every head can be treated as an independent batch element for matmul.
heads = x.view(batch, seq_len, n_heads, head_dim).transpose(1, 2)
print("split into heads:", heads.shape)   # (B, H, T, D)

# ... attention math happens here on `heads` ...

# merge back: (B, H, T, D) -> (B, T, H, D) -> (B, T, H*D)
merged = heads.transpose(1, 2).contiguous().view(batch, seq_len, hidden_size)
print("merged back:", merged.shape)
print("round-trip is lossless:", torch.equal(merged, x))

split into heads: torch.Size([2, 4, 5, 4])
merged back: torch.Size([2, 5, 16])
round-trip is lossless: True


**Where this shows up:** exactly this split -> attend -> merge pattern is
the skeleton of the `GQAAttention` module you'll write next. Getting the
order of `view` and `transpose` backwards is one of the easiest ways to
silently scramble which head owns which slice of the hidden vector — a bug
that runs without crashing but produces wrong numbers, so the parity test
against Hugging Face is what would actually catch it.

## 3. Broadcasting

When two tensors have different shapes, PyTorch tries to **broadcast** them:
it aligns shapes from the right, and any dimension of size 1 (or missing) is
stretched to match the other tensor — without actually copying data.

Rule: compare shapes right-to-left; two dims are compatible if they're equal
or one of them is 1.

In [7]:
# RMSNorm's weight (hidden_size,) broadcasts against activations (B, T, hidden_size)
weight = torch.randn(4)
x = torch.randn(2, 3, 4)
print((weight * x).shape)   # weight is broadcast across batch and seq dims

# A causal mask (T, T) broadcasts across (batch, heads, T, T) attention scores
scores = torch.randn(2, 8, 5, 5)         # (B, H, T, T)
causal_mask = torch.triu(torch.full((5, 5), float("-inf")), diagonal=1)
masked = scores + causal_mask            # (T,T) broadcasts over (B,H,*,*)
print(masked.shape)

torch.Size([2, 3, 4])
torch.Size([2, 8, 5, 5])


**Where this shows up:** RMSNorm's per-feature `weight` broadcasting over
`(batch, seq, hidden)` (you already rely on this), and the causal attention
mask broadcasting over `(batch, heads, seq, seq)` in Day 2's attention
implementation.

## 4. Indexing and slicing

Standard Python slicing works, plus PyTorch adds boolean masks and
"advanced" (tensor/array) indexing. Two patterns you'll use a lot for the
KV cache: writing into a *slice* of a preallocated tensor, and gathering
rows by an index tensor (e.g. selecting which KV head a query head reads
from, in GQA).

In [8]:
kv_cache = torch.zeros(1, 8, 100, 16)   # (batch, kv_heads, max_seq_len, head_dim) -- preallocated once

new_keys = torch.randn(1, 8, 3, 16)     # this step's new K vectors, 3 new tokens
start = 10
kv_cache[:, :, start:start + 3, :] = new_keys   # write into a slice -- no reallocation
print("wrote into cache slice, nonzero rows:", kv_cache[:, :, start:start + 3, :].abs().sum().item() > 0)

# Gathering: which KV head does each of 8 query heads read, for group size g=4, n_kv=2?
n_q, n_kv = 8, 2
group_size = n_q // n_kv
kv_head_for_query_head = torch.arange(n_q) // group_size
print("query head -> kv head map:", kv_head_for_query_head.tolist())

kv_heads_tensor = torch.arange(n_kv).float().unsqueeze(-1)  # fake per-head values, shape (n_kv, 1)
gathered = kv_heads_tensor[kv_head_for_query_head]           # (n_q, 1): index_select via fancy indexing
print("gathered per-query-head KV assignment:\n", gathered.squeeze(-1))

wrote into cache slice, nonzero rows: True
query head -> kv head map: [0, 0, 0, 0, 1, 1, 1, 1]
gathered per-query-head KV assignment:
 tensor([0., 0., 0., 0., 1., 1., 1., 1.])


**Where this shows up:** exactly the `kv_head_for_query_head` computation
above (`query_head // group_size`) is the GQA head-mapping rule from your
plan, and slice-writes into a preallocated tensor are how `kv_cache.py`
appends new tokens without reallocating memory every decode step.

## 5. Matrix multiplication: `@`, `matmul`, `bmm`, `einsum`

- **`a @ b`** / **`torch.matmul(a, b)`**: batched matrix multiply. If `a` is
  `(..., n, k)` and `b` is `(..., k, m)`, the leading `...` dims are
  broadcast and matched as independent "batches," and each matched pair of
  trailing `(n,k) @ (k,m)` matrices multiplies normally.
- **`torch.bmm`**: strictly 3D batched matmul, `(B, n, k) @ (B, k, m)`, no
  broadcasting. Slightly more explicit/restrictive than `matmul`.
- **`torch.einsum`**: express any contraction with index notation. Useful
  when a shape has many dimensions and you want the operation to be
  unambiguous to read, at some cost to unfamiliarity.

In [9]:
# Linear projection: x @ W is THE fundamental operation of every layer.
x = torch.randn(2, 5, 8)     # (batch, seq, hidden_in)
W = torch.randn(8, 16)       # (hidden_in, hidden_out)
y = x @ W                    # (2, 5, 16) -- W is broadcast as a "batch" of 1
print(y.shape)

# Attention scores: Q @ K^T per head, batched over (batch, heads)
q = torch.randn(2, 4, 5, 16)   # (B, H, T, D)
k = torch.randn(2, 4, 5, 16)   # (B, H, T, D)
scores = q @ k.transpose(-2, -1)   # (B, H, T, D) @ (B, H, D, T) -> (B, H, T, T)
print(scores.shape)

# The same thing with einsum, spelled out explicitly:
scores_einsum = torch.einsum("bhtd,bhsd->bhts", q, k)
print("einsum matches matmul:", torch.allclose(scores, scores_einsum, atol=1e-5))

torch.Size([2, 5, 16])
torch.Size([2, 4, 5, 5])
einsum matches matmul: True


**Where this shows up:** every `nn.Linear`-style projection (Q/K/V/O
projections, the SwiGLU MLP's three matrices) is a matmul, and attention
scores are exactly the batched matmul shown above. You'll see both `@` and
`einsum` in different real codebases (HF tends to use `matmul`/`@`;
some kernel-adjacent code prefers `einsum` for clarity on high-rank
tensors).

## 6. Autograd: `requires_grad`, `backward`, `no_grad`, `inference_mode`

PyTorch tracks operations on tensors with `requires_grad=True` in a graph,
so it can compute gradients via `.backward()`. **This project is about
inference, not training** — but you still need to know this because:

1. `nn.Parameter` tensors default to `requires_grad=True`, and you must
   turn gradient tracking *off* for inference, or you waste memory and time
   building a graph you'll never use `.backward()` on.
2. `torch.no_grad()` and `torch.inference_mode()` are the two ways to do
   that. `inference_mode()` is the stricter, faster modern version — prefer
   it for anything that is purely inference (which is everything in this
   project after weights are loaded).

In [10]:
x = torch.randn(3, requires_grad=True)
y = (x ** 2).sum()
y.backward()
print("gradient dy/dx = 2x:", x.grad, "expected:", 2 * x)

gradient dy/dx = 2x: tensor([-0.1730,  3.5522, -1.4830]) expected: tensor([-0.1730,  3.5522, -1.4830], grad_fn=<MulBackward0>)


In [11]:
w = nn.Parameter(torch.randn(4, 4))   # requires_grad=True by default
x = torch.randn(2, 4)

# Without no_grad: builds a graph, wastes memory, and the output tensor
# carries `requires_grad=True` even though we'll never call backward().
y_tracked = x @ w
print("tracked requires_grad:", y_tracked.requires_grad)

with torch.inference_mode():
    y_inference = x @ w
    print("inference_mode requires_grad:", y_inference.requires_grad)

# inference_mode tensors are more restricted than no_grad tensors -- e.g.
# you generally cannot mix them back into an autograd-tracked computation.
# For a serving engine this is exactly the mode you want everywhere.

tracked requires_grad: True
inference_mode requires_grad: False


**Where this shows up:** `benchmark.py` and every decode loop should wrap
the forward pass in `torch.inference_mode()`. Forgetting this is a common,
silent source of both extra memory usage and (on some ops) slower kernels,
because PyTorch is keeping bookkeeping around for a backward pass that will
never happen.

## 7. `nn.Module`, `nn.Parameter`, and buffers

You've already written one of these (`RMSNorm`). The pattern:

- Subclass `nn.Module`.
- Learnable weights are wrapped in `nn.Parameter` — this registers them so
  PyTorch's `.parameters()`, `.state_dict()`, `.to(device)`, etc. all find
  them automatically.
- Non-learnable but persistent state (constants that should move with the
  module to a device and be saved/loaded, but never get a gradient) use
  **buffers** via `self.register_buffer(name, tensor)`. RoPE's precomputed
  rotation frequencies are the canonical example, coming up next.
- `forward()` defines what happens when you call the module like a
  function: `norm(x)` actually calls `norm.__call__(x)`, which calls
  `forward` plus some bookkeeping (hooks, etc.) — this is why you define
  `forward` but call the instance directly, never `.forward()` yourself.

In [12]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # so `import model` finds ../model.py

from model import RMSNorm

norm = RMSNorm(hidden_size=8)

print("Parameters registered automatically:")
for name, p in norm.named_parameters():
    print(" ", name, p.shape, "requires_grad:", p.requires_grad)

print("\nstate_dict (what gets saved/loaded):")
print(norm.state_dict())

x = torch.randn(2, 8)
out = norm(x)   # calls __call__ -> forward
print("\noutput shape:", out.shape)

Parameters registered automatically:
  weight torch.Size([8]) requires_grad: True

state_dict (what gets saved/loaded):
OrderedDict({'weight': tensor([1., 1., 1., 1., 1., 1., 1., 1.])})

output shape: torch.Size([2, 8])


In [13]:
# Illustrating register_buffer vs nn.Parameter with a tiny example: a
# constant scale that should move with .to(device) but never train.
class ScaleByConstant(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        # Not learnable, but should be saved in state_dict and follow .to(device).
        self.register_buffer("scale", torch.arange(hidden_size).float() + 1.0)

    def forward(self, x):
        return x * self.scale

m = ScaleByConstant(4)
print("buffers:", dict(m.named_buffers()))
print("parameters (should be empty):", list(m.named_parameters()))
print("state_dict includes the buffer:", m.state_dict())

buffers: {'scale': tensor([1., 2., 3., 4.])}
parameters (should be empty): []
state_dict includes the buffer: OrderedDict({'scale': tensor([1., 2., 3., 4.])})


**Where this shows up:** every block you write from here (`RoPE`,
`GQAAttention`, `SwiGLUMLP`, the full `DecoderBlock`) is an `nn.Module`
following exactly this shape. RoPE's precomputed frequency table is a
buffer, not a parameter — it's derived from position and `head_dim`, never
learned, but must move to whatever device the model runs on.

## 8. Numerical precision: fp32 vs bf16/fp16, and why reductions get upcast

Real model weights typically ship in **bf16** (bfloat16): same 8-bit
exponent range as fp32 (so it won't overflow/underflow the way fp16 can),
but only 7 mantissa bits (vs fp32's 23), so it's much coarser. This is fine
for storing weights and doing most matmuls, but **reductions that sum many
small differences** (mean, variance, softmax normalization) can lose real
precision in bf16.

This is exactly why your `RMSNorm.forward` upcasts to `float32` before
computing `mean(x^2)`, and only casts back to the working dtype afterwards.

In [14]:
# Demonstrate the precision gap directly: summing many bf16 numbers loses
# more accuracy than summing the same numbers in fp32.
torch.manual_seed(0)
vals = torch.randn(100_000)

sum_fp32 = vals.to(torch.float32).sum()
sum_bf16 = vals.to(torch.bfloat16).sum().to(torch.float32)

print("fp32 sum:  ", sum_fp32.item())
print("bf16 sum:  ", sum_bf16.item())
print("difference:", (sum_fp32 - sum_bf16).abs().item())

fp32 sum:   -244.56166076660156
bf16 sum:   -245.0
difference: 0.4383392333984375


In [15]:
# The RMSNorm-specific version: compute mean(x^2) in bf16 vs fp32 and compare.
x = torch.randn(4096, dtype=torch.bfloat16)

mean_sq_bf16 = (x * x).mean()                       # all math stays in bf16
mean_sq_fp32 = (x.float() * x.float()).mean()       # upcast first, like your RMSNorm does

print("mean(x^2) computed in bf16:", mean_sq_bf16.item())
print("mean(x^2) computed in fp32:", mean_sq_fp32.item())
print("relative difference:", ((mean_sq_bf16.float() - mean_sq_fp32).abs() / mean_sq_fp32).item())

mean(x^2) computed in bf16: 0.99609375
mean(x^2) computed in fp32: 0.994210422039032
relative difference: 0.0018942950991913676


**Where this shows up:** this is literally the comment block in your
`RMSNorm.forward` — now you've seen the actual numeric gap it's guarding
against, not just the claim. The same upcast-then-downcast pattern reappears
in softmax (Day 2) and in the parity tests you write against Hugging Face
(if you compare in the "wrong" dtype, you'll see mismatches that aren't
bugs, just precision noise — this is why parity tests use `atol`/`rtol`,
not exact equality; see Section 18).

## 9. Softmax and masking

Attention's core operation. `F.softmax(scores, dim=-1)` turns raw scores
into a probability distribution over the last dimension. For **causal**
(autoregressive) attention, token *t* must not see tokens after it — this
is enforced by adding `-inf` to the disallowed positions *before* the
softmax, so their probability becomes exactly 0.

In [16]:
scores = torch.randn(1, 1, 5, 5)   # (batch, heads, query_pos, key_pos)

# Build a causal mask: True where key_pos > query_pos (i.e. "the future")
seq_len = 5
future = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
print("mask (True = disallowed, i.e. the future):\n", future)

masked_scores = scores.masked_fill(future, float("-inf"))
probs = F.softmax(masked_scores, dim=-1)

print("\nrow 0 (token 0 can only see itself):", probs[0, 0, 0])
print("row 4 (token 4 can see everyone):    ", probs[0, 0, 4])
print("\nprobabilities in the future are exactly zero:",
      torch.all(probs[0, 0][future] == 0).item())

mask (True = disallowed, i.e. the future):
 tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])



row 0 (token 0 can only see itself): tensor([1., 0., 0., 0., 0.])
row 4 (token 4 can see everyone):     tensor([0.4594, 0.0504, 0.1020, 0.1852, 0.2029])

probabilities in the future are exactly zero: True


**Where this shows up:** this `masked_fill(-inf) -> softmax` pattern is the
whole causal-attention mechanism from Day 2. `-inf` (not just a very
negative number) guarantees an exact zero after softmax, which matters for
the parity test against Hugging Face's reference attention.

## 10. `repeat_interleave` and `expand` for GQA's shared KV heads

Recall GQA: `n_kv` key/value heads are each shared by `group_size = n_q //
n_kv` query heads. Before computing attention, the K/V tensors need to be
"expanded" so every query head has a K/V tensor to multiply against —
**without** actually duplicating memory if you can avoid it.

- **`repeat_interleave(g, dim=d)`**: actually duplicates data `g` times
  along dimension `d`, interleaved so head `0` becomes heads `0..g-1`. Uses
  real memory — this is the "just materialize it" approach (what your
  earlier `np.repeat` example did).
- **`expand`**: creates a *view* with a dimension of size 1 stretched to
  size `n`, using stride-0 tricks — **no memory copy**. This is closer to
  what a real fused kernel does (index the same KV head from multiple query
  heads without duplicating it), though a plain `expand` alone doesn't
  reshape cleanly for matmul without a subsequent `reshape` (which forces
  the copy back) — so in practice, production kernels avoid materializing
  entirely by indexing the KV head directly per query head, but `expand`
  is the mechanism to know about for understanding why "repeat" costs
  memory and "expand" conceptually doesn't.

In [17]:
n_q, n_kv, head_dim, seq_len = 8, 2, 4, 3
group_size = n_q // n_kv

k = torch.arange(n_kv * seq_len * head_dim).reshape(1, n_kv, seq_len, head_dim).float()
print("k.shape (n_kv heads):", k.shape)

# repeat_interleave: materializes group_size copies of each KV head
k_expanded = k.repeat_interleave(group_size, dim=1)
print("k_expanded.shape (n_q heads, real copies):", k_expanded.shape)
print("uses independent memory:", k_expanded.data_ptr() != k.data_ptr())

# Check the head mapping is exactly `query_head // group_size`, matching Section 4:
for q_head in range(n_q):
    expected_kv_head = q_head // group_size
    assert torch.equal(k_expanded[0, q_head], k[0, expected_kv_head])
print("every query head's expanded K matches its assigned KV head: OK")

k.shape (n_kv heads): torch.Size([1, 2, 3, 4])
k_expanded.shape (n_q heads, real copies): torch.Size([1, 8, 3, 4])
uses independent memory: True
every query head's expanded K matches its assigned KV head: OK


In [18]:
# expand: a VIEW, no copy -- but notice the resulting stride has a 0 in it,
# which is why you can't .view() the result directly (same non-contiguous
# story as Section 2).
k_for_expand = k.unsqueeze(2)                       # (1, n_kv, 1, seq_len, head_dim)
k_view = k_for_expand.expand(1, n_kv, group_size, seq_len, head_dim)
print("expand shape:", k_view.shape, "stride:", k_view.stride())   # a 0 stride on the expanded dim
print("expand shares memory with k:", k_view.data_ptr() == k.data_ptr())

expand shape: torch.Size([1, 2, 4, 3, 4]) stride: (24, 12, 0, 4, 1)
expand shares memory with k: True


**Where this shows up:** this is the exact mechanism behind the plan's
warning in the GQA lesson — "never actually copy with `repeat`; real
kernels just index `kv_head = q_head // group_size` directly." Your Triton
paged-decode kernel in Week 3 will do that indexing inside the kernel; for
now, in a plain PyTorch implementation, `repeat_interleave` is the simple,
correct (if not maximally memory-efficient) way to make GQA attention work.

## 11. Complex numbers for RoPE: `polar`, `view_as_complex`, `view_as_real`

RoPE rotates each adjacent pair of dimensions in a query/key vector by an
angle proportional to position. The cleanest way to express "rotate a 2D
vector by angle θ" is complex multiplication: treating each pair `(x0, x1)`
as the complex number `x0 + i*x1`, multiplying by `e^{iθ} = cos θ + i sin θ`
rotates it by θ, and `|e^{iθ}| = 1` guarantees the vector's length (and
therefore the dot-product-based attention score's scale) is preserved —
only the *angle* changes.

PyTorch has native complex tensor support that makes this a one-liner
instead of hand-written rotation-matrix algebra.

**Caveat you will hit on this machine:** MPS does not support complex
tensor operations. This is exactly the kind of practical gap the plan
alludes to with the general "some ops aren't supported everywhere" theme.
The fix is simple and worth internalizing: do the complex-valued part of
RoPE on CPU, then move the (real-valued) result to `mps`.

In [19]:
# A 2D vector as a complex number, rotated by 90 degrees.
v = torch.complex(torch.tensor(1.0), torch.tensor(0.0))   # 1 + 0i, i.e. the vector (1, 0)
angle = torch.tensor(torch.pi / 2)
rotation = torch.polar(torch.tensor(1.0), angle)           # e^{i*angle}, magnitude 1

rotated = v * rotation
print("original (1,0) rotated 90 degrees ->", rotated.real.item(), rotated.imag.item())
# Expect roughly (0, 1): a 90-degree rotation sends (1,0) to (0,1).

original (1,0) rotated 90 degrees -> -4.371138828673793e-08 1.0


In [20]:
# The RoPE-style batch version: treat consecutive pairs of a vector's
# dimensions as complex numbers, then rotate every pair by a
# position-dependent angle. This previews exactly what you'll write for RoPE.
head_dim = 8
seq_len = 4

x = torch.randn(seq_len, head_dim)   # pretend this is one head's Q for `seq_len` positions

# View adjacent pairs (x[...,0],x[...,1]), (x[...,2],x[...,3]), ... as complex numbers.
x_complex = torch.view_as_complex(x.reshape(seq_len, head_dim // 2, 2))
print("x_complex.shape:", x_complex.shape)   # (seq_len, head_dim/2) complex numbers

# One rotation angle per (position, pair-index) -- RoPE's actual frequency
# schedule is more specific (Day 16), but the mechanism is this:
positions = torch.arange(seq_len).unsqueeze(1).float()       # (seq_len, 1)
pair_freqs = torch.arange(head_dim // 2).float()              # (head_dim/2,)
angles = positions * (1.0 / (10000 ** (pair_freqs / (head_dim // 2))))  # (seq_len, head_dim/2)

rotations = torch.polar(torch.ones_like(angles), angles)      # unit-magnitude complex rotations
x_rotated_complex = x_complex * rotations                     # elementwise complex multiply

x_rotated = torch.view_as_real(x_rotated_complex).reshape(seq_len, head_dim)
print("x_rotated.shape:", x_rotated.shape)   # back to a normal real tensor, same shape as input

# Rotation preserves vector length within each pair -- check on pair 0:
orig_pair0_norm = x[:, 0:2].norm(dim=-1)
rotated_pair0_norm = x_rotated[:, 0:2].norm(dim=-1)
print("length preserved per pair:", torch.allclose(orig_pair0_norm, rotated_pair0_norm, atol=1e-5))

x_complex.shape: torch.Size([4, 4])
x_rotated.shape: torch.Size([4, 8])
length preserved per pair: True


In [21]:
# The MPS caveat, shown directly:
try:
    torch.view_as_complex(torch.randn(4, 2, device=device))
    print("complex ops work on", device)
except (RuntimeError, NotImplementedError) as err:
    print(f"As expected on {device!r}, complex ops are unsupported:\n  {err}")
    print("\nFix: compute the complex rotation on CPU, then .to(device) the real result.")

complex ops work on mps


**Where this shows up:** this is the entire mechanism of `RoPE.forward`,
which you'll write next. The device caveat above is exactly the kind of
thing that costs an hour of confusing debugging if you don't know it going
in — now you do.

## 12. `register_buffer` for precomputed constants

RoPE's angle schedule (`positions * frequencies` from the cell above)
doesn't depend on the input at all — only on `head_dim`, a chosen base
(10000), and the maximum sequence length. Computing it fresh on every
forward call would be wasted work. The idiomatic fix: precompute it once in
`__init__` and store it as a **buffer**, so it's computed once, moves with
`.to(device)`, and is excluded from gradient tracking and from the
optimizer (there's nothing to train here, since it's not learned).

In [22]:
class PrecomputedAngles(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        pair_freqs = torch.arange(0, head_dim, 2).float() / head_dim
        inv_freq = 1.0 / (base ** pair_freqs)                     # (head_dim/2,)
        positions = torch.arange(max_seq_len).float()             # (max_seq_len,)
        angles = torch.outer(positions, inv_freq)                 # (max_seq_len, head_dim/2)

        # persistent=True (the default) -> saved in state_dict too.
        # Use persistent=False if you'd rather recompute it on load than store it.
        self.register_buffer("angles", angles, persistent=False)

    def forward(self, seq_len):
        return self.angles[:seq_len]   # just slice the precomputed table

pe = PrecomputedAngles(head_dim=8, max_seq_len=2048)
print("precomputed once, shape:", pe.angles.shape)
print("sliced for a seq_len=5 forward call:", pe(5).shape)
print("buffer moves with the module:", pe.to(device).angles.device)

precomputed once, shape: torch.Size([2048, 4])
sliced for a seq_len=5 forward call: torch.Size([5, 4])
buffer moves with the module: mps:0


**Where this shows up:** your `RoPE` module will precompute its angle table
this way in `__init__`, exactly like `PrecomputedAngles` above, rather than
recomputing trig functions on every single forward call.

## 13. Preallocated tensors + in-place writes: the KV-cache pattern

Naively, you might grow the KV cache with `torch.cat` every decode step —
append the new token's K/V to a running tensor. This reallocates memory on
every single step, which is disastrous for decode latency at scale.

The production pattern (which you'll build properly in `kv_cache.py`, Day
3): **preallocate** a fixed-size buffer for the maximum sequence length up
front, and **write into a slice** of it at each step. No reallocation, ever,
after the first allocation.

In [23]:
import time

max_seq_len, n_layers, n_kv_heads, head_dim = 1024, 4, 2, 16

# --- Approach A: torch.cat every step (what NOT to do for a real engine) ---
cache_cat = torch.zeros(1, n_kv_heads, 0, head_dim)
t0 = time.perf_counter()
for step in range(200):
    new_k = torch.randn(1, n_kv_heads, 1, head_dim)
    cache_cat = torch.cat([cache_cat, new_k], dim=2)   # reallocates a bigger tensor EVERY step
t_cat = time.perf_counter() - t0

# --- Approach B: preallocate once, write into a slice ---
cache_prealloc = torch.zeros(1, n_kv_heads, max_seq_len, head_dim)
t0 = time.perf_counter()
for step in range(200):
    new_k = torch.randn(1, n_kv_heads, 1, head_dim)
    cache_prealloc[:, :, step:step + 1, :] = new_k     # no reallocation
t_prealloc = time.perf_counter() - t0

print(f"torch.cat approach:    {t_cat*1000:.2f} ms for 200 steps")
print(f"preallocated approach: {t_prealloc*1000:.2f} ms for 200 steps")
print(f"speedup: {t_cat / t_prealloc:.1f}x")

torch.cat approach:    0.95 ms for 200 steps
preallocated approach: 0.34 ms for 200 steps
speedup: 2.8x


**Where this shows up:** this is the central design decision of
`kv_cache.py`. The speedup above is on tiny toy tensors on CPU/MPS; on a
real GPU with real model sizes and thousands of decode steps, the
`torch.cat` approach is far worse relative to preallocation, because it
also fragments GPU memory — one of the exact problems PagedAttention
(Week 5) was designed to solve at a larger scale (blocks instead of one
giant preallocated buffer, so memory isn't wasted reserving worst-case
length for every sequence).

## 14. Sampling primitives: `topk`, `sort`, `cumsum`, `multinomial`

Day 4's exercise (greedy, temperature, top-k, top-p, min-p sampling) is
built entirely from these tensor ops.

In [24]:
logits = torch.tensor([2.0, 1.0, 0.1, 3.0, 0.5])

# Greedy: argmax
greedy_token = torch.argmax(logits).item()
print("greedy:", greedy_token)

# Temperature: divide logits before softmax. T<1 sharpens, T>1 flattens.
for temperature in [0.5, 1.0, 2.0]:
    probs = F.softmax(logits / temperature, dim=-1)
    print(f"T={temperature}: probs =", probs.tolist())

greedy: 3
T=0.5: probs = [0.1163257360458374, 0.01574297808110714, 0.002602296182885766, 0.8595374226570129, 0.005791517440229654]
T=1.0: probs = [0.22427257895469666, 0.08250527083873749, 0.033544138073921204, 0.6096360683441162, 0.05004197731614113]
T=2.0: probs = [0.2430511862039566, 0.14741800725460052, 0.09399786591529846, 0.4007236659526825, 0.11480925232172012]


In [25]:
# Top-k: keep only the k highest-scoring tokens, zero out (i.e. -inf) the rest, then sample.
def top_k_filter(logits, k):
    values, indices = torch.topk(logits, k)
    filtered = torch.full_like(logits, float("-inf"))
    filtered[indices] = values
    return filtered

filtered = top_k_filter(logits, k=2)
print("top-2 filtered logits:", filtered)
probs = F.softmax(filtered, dim=-1)
print("resulting probs (only 2 nonzero):", probs)

sample = torch.multinomial(probs, num_samples=1)
print("sampled token index:", sample.item())

top-2 filtered logits: tensor([2., -inf, -inf, 3., -inf])
resulting probs (only 2 nonzero): tensor([0.2689, 0.0000, 0.0000, 0.7311, 0.0000])
sampled token index: 0


In [26]:
# Top-p (nucleus): keep the smallest set of highest-probability tokens whose
# cumulative probability exceeds p, then renormalize and sample among them.
def top_p_filter(logits, p):
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    sorted_probs = F.softmax(sorted_logits, dim=-1)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # Keep tokens up to and including the first one that crosses `p`.
    keep_mask = cumulative_probs <= p
    keep_mask[0] = True   # always keep at least the single highest-probability token

    filtered_sorted = torch.where(keep_mask, sorted_logits, torch.full_like(sorted_logits, float("-inf")))

    # Scatter back to original (unsorted) order.
    filtered = torch.full_like(logits, float("-inf"))
    filtered[sorted_indices] = filtered_sorted
    return filtered

filtered = top_p_filter(logits, p=0.8)
print("top-p(0.8) filtered logits:", filtered)
print("probs:", F.softmax(filtered, dim=-1))

top-p(0.8) filtered logits: tensor([-inf, -inf, -inf, 3., -inf])
probs: tensor([0., 0., 0., 1., 0.])


**Where this shows up:** this is essentially the full content of Day 4's
`sampling.py` (minus min-p, which follows the same shape: keep tokens whose
probability is at least `min_p * max_probability`). Note the deliberate
`torch.full_like(..., float("-inf"))` pattern reused from Section 9 — the
same "set disallowed entries to `-inf` before softmax" trick used for causal
masking is reused here for filtering.

## 15. Inspecting and loading real pretrained weights

Hugging Face checkpoints are ultimately just a `state_dict`: a flat mapping
from parameter name (a string like `model.layers.0.self_attn.q_proj.weight`)
to a tensor. `nn.Module.load_state_dict()` copies matching-named tensors
into your module's parameters. This is how Day 2's parity test will load
*real* Qwen3/Llama weights into *your* hand-written modules.

In [27]:
# A minimal illustration with a toy module (no download needed for this demo).
class TinyBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(4, 4, bias=False)

block_a = TinyBlock()
block_b = TinyBlock()   # freshly, independently initialized -- different weights

print("before loading, weights differ:", not torch.equal(block_a.proj.weight, block_b.proj.weight))

# This is what happens conceptually when you load a real checkpoint into your model:
block_b.load_state_dict(block_a.state_dict())
print("after load_state_dict, weights match:", torch.equal(block_a.proj.weight, block_b.proj.weight))

# `strict=False` lets you load a partial/renamed state dict -- useful when your
# module's parameter names don't exactly match Hugging Face's naming scheme,
# which you'll need to handle when wiring in real Qwen3/Llama weights.
missing_unexpected = block_b.load_state_dict({"proj.weight": torch.randn(4, 4)}, strict=False)
print(missing_unexpected)

before loading, weights differ: True
after load_state_dict, weights match: True
<All keys matched successfully>


**Where this shows up:** Day 2's instruction to "test against the Hugging
Face reference: load real weights and report max-abs-error per layer" means
exactly this — instantiate your hand-written module, `load_state_dict()`
the corresponding slice of a real checkpoint's weights into it (renaming
keys as needed to match your naming), run both your module and HF's on the
same input, and diff the outputs.

## 16. Apple Silicon / MPS: what works, what doesn't

You're developing on an Apple Silicon Mac, so MPS (`device="mps"`) is your
local accelerator. Two practical rules for this project:

1. **Not every op is implemented for MPS.** Complex tensors (Section 11) are
   one gap; you may hit others (some MPS PyTorch versions lag CPU/CUDA on
   op coverage, especially newer or unusual ops). When you hit
   `NotImplementedError`, the fix is almost always: do that one operation on
   CPU, then move the result back to `mps`.
2. **MPS is not CUDA.** The plan's GPU-hours (Weeks 2–11) assume a rented
   NVIDIA GPU for CUDA kernels, Nsight profiling, and multi-GPU work — none
   of that is available locally. Locally, your Mac is for exactly what
   Section 4 of the plan says: reading, napkin math, `perf_model.py`, and
   *small* CPU/MPS experiments — including everything in this notebook and
   Week 1's model code (small enough to run on CPU or MPS for correctness,
   even though real benchmarking happens on rented GPUs later).

In [28]:
# A general-purpose fallback pattern worth keeping handy:
def safe_to_device(tensor, device, cpu_only_ops=()):
    # Illustrative helper: run known-unsupported ops on CPU, then move over.
    return tensor.to(device)

def rope_style_op(x, device):
    # Do the complex-number part on CPU regardless of target device...
    x_cpu = x.to("cpu")
    pairs = torch.view_as_complex(x_cpu.reshape(*x_cpu.shape[:-1], -1, 2))
    rotated = torch.view_as_real(pairs * 1.0).reshape(x_cpu.shape)
    # ...then move the REAL-valued result to the actual target device.
    return rotated.to(device)

out = rope_style_op(torch.randn(2, 8), device)
print("computed on CPU where needed, final device:", out.device)

computed on CPU where needed, final device: mps:0


**Where this shows up:** exactly this CPU-fallback shape is what your
`RoPE` module will need on this machine. It's a two-line adjustment, but
only if you know to expect it — otherwise it looks like your code is
broken when really it's just an MPS op-coverage gap.

## 17. Testing numerics: `torch.testing.assert_close`

You already used this in `tests/test_model.py`. Two things worth being
explicit about:

- Never compare floating-point tensors with `==`. Different but
  mathematically-equivalent computation orders (e.g. fused vs unfused ops,
  or different dtypes) produce tiny differences purely from floating-point
  rounding — this is expected, not a bug.
- `torch.testing.assert_close(a, b, atol=..., rtol=...)` checks
  `|a - b| <= atol + rtol * |b|` elementwise. `atol` (absolute tolerance)
  dominates when values are near zero; `rtol` (relative tolerance) dominates
  for larger values. Picking these isn't arbitrary — it should reflect the
  dtype's actual precision (e.g. bf16 needs much looser tolerances than
  fp32).

In [29]:
a = torch.tensor([1.0, 100.0, 1e-8])
b = a + torch.tensor([1e-7, 1e-3, 1e-9])   # tiny absolute noise, scaled differently per element

try:
    torch.testing.assert_close(a, b, atol=0.0, rtol=0.0)
except AssertionError as err:
    print("Exact equality fails, as expected:\n", str(err)[:300])

# A reasonable tolerance for fp32 comparisons:
torch.testing.assert_close(a, b, atol=1e-6, rtol=1e-3)
print("\nPasses with realistic atol/rtol.")

Exact equality fails, as expected:
 Tensor-likes are not equal!

Mismatched elements: 3 / 3 (100.0%)
Greatest absolute difference: 0.00099945068359375 at index (1,)
Greatest relative difference: 0.09090910106897354 at index (2,)

Passes with realistic atol/rtol.


**Where this shows up:** every parity test in this project, current and
future — including the ones already in `tests/test_model.py`. When Day 2
compares your attention output to Hugging Face's, expect to loosen
tolerances specifically when comparing in bf16, and to explain (per the
plan's "explain the predicted-vs-measured gap" rule) *why* a given tolerance
is appropriate rather than picking one that happens to make the test pass.

## 18. A preview: `torch.compile` and CUDA graphs

These belong properly to Week 2 (Day 13) and require an NVIDIA GPU to show
their real benefit, but it's worth knowing the API shape now since you'll
use it soon:

- **`torch.compile(model)`**: traces your model and JIT-compiles it (via
  TorchInductor) into fused, more efficient kernels. Mostly a free speedup
  for GPU workloads; on MPS/CPU the benefit is much smaller or absent, and
  op coverage can be incomplete — treat it as "try it on the GPU rental,
  don't expect much locally."
- **CUDA graphs**: capture an entire sequence of GPU kernel launches once,
  then "replay" the whole captured graph with a single launch. This
  specifically targets the **CPU-launch-overhead** problem the plan
  describes for small-batch decode (Day 13): at batch size 1, the GPU
  finishes each tiny matmul faster than the CPU can launch the *next* one,
  so decode becomes launch-bound rather than compute-bound. CUDA graphs are
  an NVIDIA/CUDA-specific concept — there's no MPS equivalent to try
  locally.

In [30]:
# The torch.compile API, shown on CPU/MPS just so the shape is familiar --
# do not expect a meaningful speedup here; the real test happens on a rented GPU.
class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 256)
        self.fc2 = nn.Linear(256, 64)

    def forward(self, x):
        return self.fc2(F.silu(self.fc1(x)))

m = TinyMLP().to(device)
try:
    compiled = torch.compile(m)
    x = torch.randn(8, 64, device=device)
    with torch.inference_mode():
        out = compiled(x)
    print("torch.compile ran successfully on", device, "-> output shape", out.shape)
except Exception as err:
    print(f"torch.compile had trouble on {device!r} (expected -- full support is GPU/Linux-first):\n{err}")

W0925 01:57:10.325000 16395 torch/_inductor/utils.py:1806] [0/0] Not enough SMs to use max_autotune_gemm mode


torch.compile ran successfully on mps -> output shape torch.Size([8, 64])


**Where this shows up:** Day 13's exercise ("capture the decode step in a
CUDA graph and try `torch.compile`. Measure step time at batch 1 before and
after.") is exactly the API shown above, run for real on a rented NVIDIA
GPU where both mechanisms are fully supported and their benefit is
measurable.

## Summary: concept -> where it's used

| Concept | First used in |
|---|---|
| Tensors, dtype, device | Everywhere, starting with `RMSNorm` (already done) |
| `view`/`transpose`/`contiguous` | GQA attention's split/merge heads (Day 2) |
| Broadcasting | `RMSNorm` weight (done), causal mask (Day 2) |
| Indexing, slice-writes | `kv_cache.py` (Day 3) |
| `matmul`/`einsum` | Every projection and attention score (Day 2) |
| `inference_mode` | `benchmark.py`, every decode loop |
| `nn.Module`/`Parameter`/buffer | `RMSNorm` (done), `RoPE`, `GQAAttention`, `SwiGLUMLP` (next) |
| fp32 upcast for reductions | `RMSNorm` (done), softmax, parity tolerances |
| softmax + masking | Causal attention (Day 2) |
| `repeat_interleave` | GQA's shared KV heads (Day 2) |
| `view_as_complex`/`polar` | `RoPE` (next) |
| `register_buffer` | `RoPE`'s precomputed angles (next) |
| Preallocated slice-writes | `kv_cache.py` (Day 3) |
| `topk`/`sort`/`cumsum`/`multinomial` | `sampling.py` (Day 4) |
| `state_dict`/`load_state_dict` | Loading real HF weights for parity tests (Day 2) |
| MPS fallback pattern | `RoPE` on this machine specifically |
| `assert_close`, atol/rtol | Every test in `tests/` |
| `torch.compile`, CUDA graphs | Day 13, on a rented GPU |

Next step in the project: implement **RoPE** in `model.py`, using Sections
11–13 directly.